In [9]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [8]:
df = pd.read_excel(r'Datasets/Data5_WT2025.xlsx', index_col=0, parse_dates=False)
df.index = pd.to_datetime(df.index, format='%Y%m').to_period('M')
df.drop(columns=['Unnamed: 2', 'Unnamed: 9'], inplace=True)
df['AKREX_excess'] = df['AKREX'] - df['RF']
df = df.rename(columns={'Mkt-RF' : 'Mkt'})


FileNotFoundError: [Errno 2] No such file or directory: 'Datasets/Data5_WT2025.xlsx'

In [ ]:
df

,AKREX,Mkt,SMB,HML,MOM,LIQ,RF,SMALL LoBM,SMALL medBM,SMALL HiBM,BIG LoBM,BIG medBM,BIG HiBM,BOND,INTERNATIONAL,AKREX_excess
DATE,,,,,,,,,,,,,,,,
2009-10,-0.004990,-0.0259,-0.0435,-0.0421,0.0261,0.077705,0.0000,-0.069838,-0.063198,-0.093979,-0.004418,-0.027689,-0.064484,-0.003575,-0.012947,-0.004990
2009-11,0.016048,0.0556,-0.0240,-0.0034,0.0030,-0.002845,0.0000,0.020214,0.019394,0.041184,0.066381,0.047671,0.038618,0.018977,0.017507,0.016048
2009-12,0.002962,0.0275,0.0605,-0.0016,0.0301,0.022835,0.0001,0.084759,0.080740,0.083772,0.019456,0.031202,0.017169,-0.037937,0.013590,0.002862
2010-01,-0.017717,-0.0336,0.0040,0.0043,-0.0540,-0.011057,0.0000,-0.037834,-0.030201,-0.026859,-0.042125,-0.020265,-0.044504,0.020794,-0.044380,-0.017717
2010-02,0.022044,0.0340,0.0119,0.0322,0.0374,-0.007559,0.0000,0.033733,0.041291,0.074536,0.030580,0.029082,0.054191,0.002013,-0.008785,0.022044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08,0.001860,-0.0239,-0.0320,-0.0108,0.0377,-0.042066,0.0045,-0.078782,-0.055124,-0.054211,-0.008988,-0.028145,-0.055092,-0.009085,-0.041012,-0.002640
2023-09,-0.052729,-0.0524,-0.0249,0.0145,0.0024,-0.040537,0.0043,-0.066692,-0.059003,-0.060829,-0.054008,-0.027017,-0.030848,-0.033463,-0.036934,-0.057029
2023-10,-0.024500,-0.0318,-0.0388,0.0019,0.0168,0.000784,0.0047,-0.091302,-0.060649,-0.062204,-0.017960,-0.036388,-0.043317,-0.024254,-0.041034,-0.029200


## Question 1 : 5 - Factor Regression

In [ ]:
def factor_regression(returns, period):

    #Define the period for the regression
    returns = returns.loc[period]
    # Define the regression formula
    formula = 'AKREX_excess ~ Mkt + SMB + HML + MOM + LIQ'
    
    # Fit the regression model
    model = smf.ols(formula=formula, data=returns).fit()
    
    # Get the summary of the regression results
    summary = model.summary()
    return summary, model

In [ ]:
question_1_results = factor_regression(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))
question_1_results[0].tables[1]


,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,0.0021,0.001,1.457,0.147,-0.001,0.005
Mkt,0.8933,0.034,25.962,0.000,0.825,0.961
SMB,-0.1258,0.059,-2.123,0.035,-0.243,-0.009
HML,-0.1684,0.045,-3.752,0.000,-0.257,-0.080
MOM,0.0132,0.042,0.313,0.754,-0.070,0.096
LIQ,-0.0479,0.044,-1.079,0.282,-0.135,0.040


In [ ]:
question_2a_results = factor_regression(returns = df, period = pd.period_range(start='2009-10', end='2016-10', freq='M'))
question_2b_results = factor_regression(returns = df, period = pd.period_range(start='2016-11', end='2023-12', freq='M'))
question_2a_results[0].tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,0.0039,0.002,2.372,0.020,0.001,0.007
Mkt,0.7347,0.045,16.402,0.000,0.646,0.824
SMB,0.0911,0.075,1.216,0.227,-0.058,0.240
HML,-0.1637,0.082,-2.000,0.049,-0.327,-0.001
MOM,-0.0308,0.055,-0.559,0.578,-0.141,0.079
LIQ,-0.0266,0.059,-0.448,0.656,-0.145,0.092


In [ ]:
question_2b_results[0].tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,0.0010,0.002,0.472,0.638,-0.003,0.005
Mkt,0.9840,0.050,19.797,0.000,0.885,1.083
SMB,-0.2361,0.088,-2.697,0.009,-0.410,-0.062
HML,-0.1616,0.057,-2.847,0.006,-0.275,-0.049
MOM,0.0400,0.064,0.628,0.532,-0.087,0.167
LIQ,-0.0605,0.062,-0.968,0.336,-0.185,0.064


## Question 3 : Sharpe & IR 

In [ ]:
def Sharpe_Ratio(returns, period):
    #Define period
    returns = returns.loc[period]

    #Calculate excess returns mean and variance

    excess_returns = returns['AKREX_excess'].mean()
    volatility = returns['AKREX_excess'].std()
    
    #Calculate Sharpe Ratio
    
    return (excess_returns / volatility).round(3)


In [ ]:
Q1_Sharpe = Sharpe_Ratio(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))
Q2a_Sharpe = Sharpe_Ratio(returns = df, period = pd.period_range(start='2009-10', end='2016-10', freq='M'))
Q2b_Sharpe = Sharpe_Ratio(returns = df, period = pd.period_range(start='2016-11', end='2023-12', freq='M'))
Q1_Sharpe, Q2a_Sharpe, Q2b_Sharpe

(np.float64(0.283), np.float64(0.374), np.float64(0.237))

In [ ]:
Q1_idio_st = question_1_results[1].resid.std(ddof=1)
Q2a_idio_st = question_2a_results[1].resid.std(ddof=1)
Q2b_idio_st = question_2b_results[1].resid.std(ddof=1)

In [ ]:
def Information_Ratio(returns, periods):

    #Run Regression
    results, model = factor_regression(returns = returns, period = periods)

    #Calculate Information Ratio
    idio_st = model.resid.std(ddof=1)
    alpha = model.params['Intercept']

    #Calculate Sharpe Ratio
    Sharpe = returns['AKREX_excess'].loc[periods].mean() / returns['AKREX_excess'].loc[periods].std(ddof=1)

    return Sharpe, alpha / idio_st

In [ ]:
Oct_2009_2016 = Information_Ratio(returns = df, periods = pd.period_range(start='2009-10', end='2023-12', freq='M'))
Nov_2016_Dec_2023 = Information_Ratio(returns = df, periods = pd.period_range(start='2016-11', end = '2023-12', freq='M'))

In [ ]:
#Sharpe Ratio and IR
Oct_2009_2016, Nov_2016_Dec_2023
#Maximum Optimal Sharpe
Sharpe_Period_1 = np.sum([i**i for i in Oct_2009_2016])
Sharpe_Period_2 = np.sum([i**i for i in Nov_2016_Dec_2023])
Sharpe_Period_1, Sharpe_Period_2

(np.float64(1.4767198207481522), np.float64(1.564794296339348))

## Henriksson and Merton : Market Timing

In [ ]:
df['Indicator'] = np.where(df['Mkt'] > df['RF'], 1, 0)
df['TimingFactor'] = df['Indicator'] * df['Mkt']
df.columns = df.columns.str.replace(' ','_')

In [ ]:
def market_timing_regression(returns, period):
    #Define the period for the regression
    returns = returns.loc[period]

    # Define the regression formula
    formula = 'AKREX_excess ~ Mkt + TimingFactor'

    # Fit the regression model
    model = smf.ols(formula=formula, data=returns).fit()
    # Get the summary of the regression results
    results = model.summary()

    return model, results

market_timing  = market_timing_regression(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))

In [ ]:
market_timing[1]

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           AKREX_excess   R-squared:                       0.801
Model:                            OLS   Adj. R-squared:                  0.799
Method:                 Least Squares   F-statistic:                     338.5
Date:                Fri, 04 Apr 2025   Prob (F-statistic):           1.17e-59
Time:                        14:04:24   Log-Likelihood:                 438.36
No. Observations:                 171   AIC:                            -870.7
Df Residuals:                     168   BIC:                            -861.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        0.0014      0.002      0.583      0.561      -0.003       0.006
Mkt              0.8006      0.064     12.474      0.000       0.674       0.927
TimingFactor     0.0853      0.103      0.829      0.408      -0.118       0.288
==============================================================================
Omnibus:                        1.136   Durbin-Watson:                   2.059
Prob(Omnibus):                  0.567   Jarque-Bera (JB):                0.761
Skew:                          -0.078   Prob(JB):                        0.683
Kurtosis:                       3.288   Cond. No.                         82.0
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Style Analysis

In [3]:
def unconstrained_style_analysis(returns, period):

    returns = returns.loc[period]
    # Define the regression formula
    formula = 'AKREX_excess ~ SMALL_HiBM + SMALL_LoBM + SMALL_medBM + BIG_medBM + BIG_HiBM + BIG_LoBM + BOND + INTERNATIONAL'
    # Fit the regression model
    model = smf.ols(formula=formula, data=returns).fit()
    # Get the summary of the regression results
    results = model.summary()

    return results, model

q5_unconstrained_results, q5_unconstrained_model = unconstrained_style_analysis(returns = df, period = pd.period_range(start='2009-10', end='2023-12', freq='M'))

q5_unconstrained_df = pd.concat([pd.Series({'R^2': q5_unconstrained_model.rsquared * 100})
                     , q5_unconstrained_model.params * 100], axis = 0)
q5_unconstrained_df

NameError: name 'df' is not defined

In [ ]:
def constrained_style_analysis(returns, period):

    returns = returns.loc[period]
    # Define the regression formula (constrained so we set -1 for no intercept)
    formula = 'AKREX_excess ~ SMALL_HiBM + SMALL_LoBM + BIG_HiBM + BIG_LoBM + BOND + INTERNATIONAL -1'
    
    #Define Constraint
    constraint = 'SMALL_HiBM + SMALL_LoBM + BIG_HiBM + BIG_LoBM + BOND + INTERNATIONAL = 1'
    # Fit the regression model
    model = smf.glm(formula=formula, data=returns, family=sm.families.Gaussian(sm.families.links.Identity())).fit_constrained(constraint)

    return model, model.summary()

q5_model_constrained, q5_results_constrained = constrained_style_analysis(returns = df, period= pd.period_range(start='2009-10', end='2023-12', freq='M'))

q5_constrained_df = pd.concat([pd.Series({'R^2' : q5_model_constrained.pseudo_rsquared()* 100})
                     , q5_model_constrained.params * 100], axis = 0)
q5_constrained_df

R^2              99.089104
SMALL_HiBM      -13.540235
SMALL_LoBM        8.634991
BIG_HiBM         23.240360
BIG_LoBM         66.124220
BOND             17.271622
INTERNATIONAL    -1.730958
dtype: float64

In [ ]:
from scipy.optimize import minimize
n_factors = 8 
#Constraints must sum to 1
constraints = [
    {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}
]
bounds = [(None, None)] * n_factors #No Lower or Outer Bound on weights


    
def quadratic_programming_style_analysis(returns, period, n_factors):

    #Initial Guess
    guess_0 = np.ones(n_factors) / n_factors

    #Factors
    factors = ['SMALL_HiBM', 'SMALL_LoBM', 'BIG_HiBM', 'BIG_LoBM', 'BOND', 'INTERNATIONAL', 'Mkt']

    #Objective Function
    def objective(returns, weights, factors):
    
        residuals = returns['AKREX_excess'] - np.dot(returns[factors], weights)

        return np.sum(residuals ** 2)

    result = minimize(objective, guess_0, args=(returns, )

IndentationError: expected an indented block (4267388655.py, line 9)

R^2              99.089104
SMALL_HiBM      -13.540235
SMALL_LoBM        8.634991
BIG_HiBM         23.240360
BIG_LoBM         66.124220
BOND             17.271622
INTERNATIONAL    -1.730958
dtype: float64

R^2              83.369307
Intercept         0.074789
SMALL_HiBM      -12.600433
SMALL_LoBM        8.097944
BIG_HiBM         25.606383
BIG_LoBM         63.616554
BOND             31.730158
INTERNATIONAL    -0.030628
dtype: float64

In [ ]:
df.head()

,AKREX,Mkt,SMB,HML,MOM,LIQ,RF,SMALL LoBM,SMALL medBM,SMALL HiBM,BIG LoBM,BIG medBM,BIG HiBM,BOND,INTERNATIONAL,AKREX_excess,Indicator,TimingFactor
DATE,,,,,,,,,,,,,,,,,,
2009-10,-0.004990,-0.0259,-0.0435,-0.0421,0.0261,0.077705,0.0000,-0.069838,-0.063198,-0.093979,-0.004418,-0.027689,-0.064484,-0.003575,-0.012947,-0.004990,0,-0.0000
2009-11,0.016048,0.0556,-0.0240,-0.0034,0.0030,-0.002845,0.0000,0.020214,0.019394,0.041184,0.066381,0.047671,0.038618,0.018977,0.017507,0.016048,1,0.0556
2009-12,0.002962,0.0275,0.0605,-0.0016,0.0301,0.022835,0.0001,0.084759,0.080740,0.083772,0.019456,0.031202,0.017169,-0.037937,0.013590,0.002862,1,0.0275
2010-01,-0.017717,-0.0336,0.0040,0.0043,-0.0540,-0.011057,0.0000,-0.037834,-0.030201,-0.026859,-0.042125,-0.020265,-0.044504,0.020794,-0.044380,-0.017717,0,-0.0000
2010-02,0.022044,0.0340,0.0119,0.0322,0.0374,-0.007559,0.0000,0.033733,0.041291,0.074536,0.030580,0.029082,0.054191,0.002013,-0.008785,0.022044,1,0.0340
